In [5]:
import holidays
import numpy
import numpy as np
import pandas
import config

UK_HOLIDAYS = pandas.to_datetime(list(holidays.country_holidays('UK', years=range(2011, 2026)).keys()))

raw_feature_name, target_feature_name = zip(*[
    ('Date', 'date'),
    ('Time', 'time'),
    ('Ozone', 'O3'),
    ('Nitric oxide', 'NO'),
    ('Nitrogen dioxide', 'NO2'),
    ('Carbon monoxide', 'CO'),
    ('Modelled Wind Direction', 'wind_dir'),
    ('Modelled Wind Speed', 'wind_speed'),
    ('Modelled Temperature', 'temp'),
    ('PM10 particulate matter (Hourly measured)', 'PM10'),
    ('PM2.5 particulate matter (Hourly measured)', 'PM2.5')
])

def get_day_category(date, holiday_dates):
    next_date = date + pandas.Timedelta(days=1)
    is_off_day = lambda d: is_weekend(d) | is_holiday(d, holiday_dates)

    return pandas.Categorical(numpy.select(
        [is_off_day(date), is_off_day(next_date)],
        [1, 2],
        default=0
    ))

def add_day_category(df, holiday_dates):
    category = get_day_category(df['date'], holiday_dates)
    df.insert(df.columns.get_loc('date') + 1, 'day_category', category)
    return df


def is_weekend(date):
    return date.dt.day_name().isin(['Saturday', 'Sunday'])


def is_holiday(date, country_holidays):
    return date.isin(country_holidays)


def apply_min_max(df, exclude=None):
    x = df.select_dtypes(include='number')
    if exclude:
        x = x.drop(columns=exclude, errors='ignore')
    df[x.columns] = (2 * (x - x.min()) / (x.max() - x.min()) - 1).round(4)
    return df


def process(csv_year):
    print(f"Processing {csv_year}")
    return (
        pandas.read_csv(
            f"{config.raw_csv}/{csv_year}.csv",
            na_values=['No data'],
            parse_dates=['Date'],
            skiprows=config.rows_to_skip,
            skipfooter=1,
            usecols=raw_feature_name,
            engine='python'
        )
        .rename(columns=dict(zip(raw_feature_name, target_feature_name)))
        .assign(time=lambda df: df['time'].str[:2].astype(int))
        .pipe(add_day_category, UK_HOLIDAYS)
        .assign(wind_dir_sin=lambda df: np.sin(np.radians(df['wind_dir'])).round(4),
                wind_dir_cos=lambda df: np.cos(np.radians(df['wind_dir'])).round(4))
        .drop(columns=['wind_dir'])
    )

In [6]:
combined_raw = (
    pandas.concat([process(year) for year in range(config.start_year, config.end_year + 1)])
    .sort_values(['date', 'time'])
)
combined_raw.to_csv(config.unscaled_csv, index=False)

Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
Processing 2025


In [7]:
min_max_values = pandas.DataFrame({
    'min': combined_raw.min(numeric_only=True),
    'max': combined_raw.max(numeric_only=True)
})
min_max_values.to_csv(config.min_max_csv)

In [8]:
combined_scaled = (combined_raw
                   .drop(columns=['CO'])
                   .pipe(apply_min_max, exclude=['wind_dir_sin', 'wind_dir_cos', 'day_category', 'time'])
                   .dropna())

combined_scaled.to_csv(config.scaled_csv, index=False)

In [40]:
features = ['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp', 'wind_dir_sin', 'wind_dir_cos']

def make_blocks(df, input_hours, forecast_hours, day_category):
    all_hours = input_hours + forecast_hours

    filtered = df[
        (df['day_category'] == day_category) &
        (df['time'].isin(all_hours))
    ]

    inputs = []
    outputs = []
    block_dates = []

    for date, group in filtered.groupby('date'):
        if len(group) != len(all_hours):
            continue

        sorted_group = group.sort_values('time')
        n_input = len(input_hours)
        inputs.append(sorted_group[features].iloc[:n_input].values.flatten())
        outputs.append(sorted_group[features].iloc[n_input:].values.flatten())
        block_dates.append(date)

    inputs = np.array(inputs)
    outputs = np.array(outputs)
    block_dates = np.array(block_dates)

    return inputs, outputs, block_dates

In [48]:
def knn_predict(current_input, date, inputs, outputs, block_dates, k=5):
    history_mask = block_dates < pandas.Timestamp(date)
    history_inputs = inputs[history_mask]
    history_outputs = outputs[history_mask]

    distances = np.linalg.norm(history_inputs - current_input, axis=1)
    nearest = np.argsort(distances)[:k]
    return history_outputs[nearest].mean(axis=0)

In [49]:
input_hours = [6, 7, 8]
forecast_hours = [9, 10, 11, 12, 13]

inputs, outputs, block_dates = make_blocks(combined_scaled, input_hours, forecast_hours, day_category=0)
print(f"Blokų: {len(inputs)}, input: {inputs.shape[1]}, output: {outputs.shape[1]}")

test_mask = np.array([d.year == 2025 for d in block_dates])
test_inputs = inputs[test_mask]
test_outputs = outputs[test_mask]
test_dates = block_dates[test_mask]

predictions = np.array([
    knn_predict(test_inputs[i], test_dates[i], inputs, outputs, block_dates, k=5)
    for i in range(len(test_inputs))
])

rmse = np.sqrt(((predictions - test_outputs) ** 2).mean(axis=0))

Input valandos: [6, 7, 8], Forecast valandos: [9, 10, 11, 12, 13]
Day category: 0
Blokų: 1931
Input shape: (1931, 27)
Output shape: (1931, 45)
Datos: 2012-01-09 00:00:00 — 2025-12-30 00:00:00
Blokų: 1931, input: 27, output: 45


In [43]:
pm25_idx = features.index('PM2.5')
print("\nPM2.5 prognozės tikslumas:")
pm25_rmse = []
for h_idx, hour in enumerate(forecast_hours):
    idx = h_idx * len(features) + pm25_idx
    pm25_rmse.append(rmse[idx])
    print(f"  Valanda {hour}: RMSE {rmse[idx]:.4f}")
print(f"PM2.5 vidutinis RMSE: {np.mean(pm25_rmse):.4f}")



PM2.5 prognozės tikslumas:
  Valanda 9: RMSE 0.0885
  Valanda 10: RMSE 0.0862
  Valanda 11: RMSE 0.0916
  Valanda 12: RMSE 0.0999
  Valanda 13: RMSE 0.0946
PM2.5 vidutinis RMSE: 0.0921


In [44]:
params = pandas.read_csv(config.min_max_csv, index_col=0)
pm25_min = params.loc['PM2.5', 'min']
pm25_max = params.loc['PM2.5', 'max']

def unscale(scaled_value):
    return (scaled_value + 1) * (pm25_max - pm25_min) / 2 + pm25_min

for i in range(len(forecast_hours)):
    print(f"\n{test_dates[i]}:")
    for h_idx, hour in enumerate(forecast_hours):
        idx = h_idx * len(features) + pm25_idx
        actual = unscale(test_outputs[i][idx])
        predicted = unscale(predictions[i][idx])
        print(f"  Valanda {hour}: prognozė={predicted:.1f} µg/m³, realiai={actual:.1f} µg/m³")


2025-01-06 00:00:00:
  Valanda 9: prognozė=7.1 µg/m³, realiai=2.0 µg/m³
  Valanda 10: prognozė=7.0 µg/m³, realiai=2.0 µg/m³
  Valanda 11: prognozė=6.7 µg/m³, realiai=-1.0 µg/m³
  Valanda 12: prognozė=6.3 µg/m³, realiai=3.0 µg/m³
  Valanda 13: prognozė=8.1 µg/m³, realiai=3.0 µg/m³

2025-01-07 00:00:00:
  Valanda 9: prognozė=8.4 µg/m³, realiai=6.0 µg/m³
  Valanda 10: prognozė=9.2 µg/m³, realiai=5.0 µg/m³
  Valanda 11: prognozė=9.0 µg/m³, realiai=2.0 µg/m³
  Valanda 12: prognozė=10.2 µg/m³, realiai=4.0 µg/m³
  Valanda 13: prognozė=12.7 µg/m³, realiai=5.0 µg/m³

2025-01-08 00:00:00:
  Valanda 9: prognozė=13.6 µg/m³, realiai=8.0 µg/m³
  Valanda 10: prognozė=12.3 µg/m³, realiai=8.0 µg/m³
  Valanda 11: prognozė=17.6 µg/m³, realiai=11.0 µg/m³
  Valanda 12: prognozė=16.7 µg/m³, realiai=11.0 µg/m³
  Valanda 13: prognozė=15.9 µg/m³, realiai=10.0 µg/m³

2025-01-09 00:00:00:
  Valanda 9: prognozė=9.8 µg/m³, realiai=12.0 µg/m³
  Valanda 10: prognozė=9.1 µg/m³, realiai=15.0 µg/m³
  Valanda 11: progn

In [45]:
pm25_rmse_real = []
for h_idx, hour in enumerate(forecast_hours):
    idx = h_idx * len(features) + pm25_idx
    actual = unscale(test_outputs[:, idx])
    predicted = unscale(predictions[:, idx])
    rmse_real = np.sqrt(((predicted - actual) ** 2).mean())
    pm25_rmse_real.append(rmse_real)
    print(f"Valanda {hour}: RMSE {rmse_real:.1f} µg/m³")

print(f"Vidutinis RMSE: {np.mean(pm25_rmse_real):.1f} µg/m³")

Valanda 9: RMSE 5.9 µg/m³
Valanda 10: RMSE 5.7 µg/m³
Valanda 11: RMSE 6.1 µg/m³
Valanda 12: RMSE 6.6 µg/m³
Valanda 13: RMSE 6.3 µg/m³
Vidutinis RMSE: 6.1 µg/m³


In [46]:
train_mask = ~test_mask
naive = unscale(outputs[train_mask][:, [h * len(features) + pm25_idx for h in range(len(forecast_hours))]].mean(axis=0))
print(f"\nNaive vidurkis per valandas: {naive}")


Naive vidurkis per valandas: [16.80851982 16.70806628 16.43286778 16.06184604 15.93899735]


In [20]:
for h_idx, hour in enumerate(forecast_hours):
    idx = h_idx * len(features) + pm25_idx
    actual = unscale(test_outputs[:, idx])

    # KNN
    knn_pred = unscale(predictions[:, idx])
    knn_rmse = np.sqrt(((knn_pred - actual) ** 2).mean())

    # Naive — visada prognozuoja istorinį vidurkį
    train_vals = unscale(outputs[~test_mask][:, idx])
    naive_pred = train_vals.mean()
    naive_rmse = np.sqrt(((naive_pred - actual) ** 2).mean())

    print(f"Valanda {hour}: KNN={knn_rmse:.1f} µg/m³  Naive={naive_rmse:.1f} µg/m³")

Valanda 9: KNN=5.9 µg/m³  Naive=11.9 µg/m³
Valanda 10: KNN=5.7 µg/m³  Naive=11.3 µg/m³
Valanda 11: KNN=6.1 µg/m³  Naive=10.9 µg/m³
Valanda 12: KNN=6.6 µg/m³  Naive=10.2 µg/m³
Valanda 13: KNN=6.3 µg/m³  Naive=9.7 µg/m³


In [47]:
# rasti dienas kur realus PM2.5 buvo aukštas
for i in range(len(test_dates)):
    idx_9 = 0 * len(features) + pm25_idx
    actual_9 = unscale(test_outputs[i][idx_9])

    if actual_9 > 20:  # aukštas PM2.5
        print(f"\n{test_dates[i]}:")
        for h_idx, hour in enumerate(forecast_hours):
            idx = h_idx * len(features) + pm25_idx
            actual = unscale(test_outputs[i][idx])
            predicted = unscale(predictions[i][idx])
            print(f"  Valanda {hour}: prognozė={predicted:.1f} µg/m³, realiai={actual:.1f} µg/m³")


2025-01-22 00:00:00:
  Valanda 9: prognozė=14.1 µg/m³, realiai=29.0 µg/m³
  Valanda 10: prognozė=15.0 µg/m³, realiai=27.0 µg/m³
  Valanda 11: prognozė=15.7 µg/m³, realiai=17.0 µg/m³
  Valanda 12: prognozė=13.9 µg/m³, realiai=24.0 µg/m³
  Valanda 13: prognozė=11.0 µg/m³, realiai=27.0 µg/m³

2025-02-06 00:00:00:
  Valanda 9: prognozė=12.3 µg/m³, realiai=21.0 µg/m³
  Valanda 10: prognozė=16.5 µg/m³, realiai=19.0 µg/m³
  Valanda 11: prognozė=14.4 µg/m³, realiai=15.0 µg/m³
  Valanda 12: prognozė=12.8 µg/m³, realiai=10.0 µg/m³
  Valanda 13: prognozė=14.5 µg/m³, realiai=5.0 µg/m³

2025-02-10 00:00:00:
  Valanda 9: prognozė=32.6 µg/m³, realiai=36.0 µg/m³
  Valanda 10: prognozė=33.9 µg/m³, realiai=37.0 µg/m³
  Valanda 11: prognozė=29.7 µg/m³, realiai=33.0 µg/m³
  Valanda 12: prognozė=28.8 µg/m³, realiai=35.0 µg/m³
  Valanda 13: prognozė=28.6 µg/m³, realiai=30.0 µg/m³

2025-02-11 00:00:00:
  Valanda 9: prognozė=23.3 µg/m³, realiai=29.0 µg/m³
  Valanda 10: prognozė=27.4 µg/m³, realiai=33.0 µg/m³

In [50]:
def predict_day(current_input, date, inputs, outputs, block_dates, k=5):
    history_mask = block_dates < pandas.Timestamp(date)

    distances = np.linalg.norm(inputs[history_mask] - current_input, axis=1)
    nearest = np.argsort(distances)[:k]
    return outputs[history_mask][nearest].mean(axis=0)

# paruoši vieną kartą
inputs, outputs, block_dates = make_blocks(combined_scaled, [4,5,6,7,8], [9,10,11,12,13], day_category=0)

# tada kvieti greitai
date = '2025-03-06'
day_data = combined_scaled[
    (combined_scaled['date'] == date) &
    (combined_scaled['time'].isin([4,5,6,7,8]))
].sort_values('time')

current_input = day_data[features].values.flatten()
prediction = knn_predict(current_input, '2025-03-06', inputs, outputs, block_dates, k=5)

# actual — tos dienos realūs duomenys
actual_data = combined_scaled[
    (combined_scaled['date'] == date) &
    (combined_scaled['time'].isin([9,10,11,12,13]))
].sort_values('time')

pm25_idx = features.index('PM2.5')
for h_idx, hour in enumerate([9,10,11,12,13]):
    idx = h_idx * len(features) + pm25_idx
    predicted = unscale(prediction[idx])
    actual = unscale(actual_data[actual_data['time'] == hour]['PM2.5'].values[0])
    print(f"Valanda {hour}: prognozė={predicted:.1f} µg/m³, realiai={actual:.1f} µg/m³")

Input valandos: [4, 5, 6, 7, 8], Forecast valandos: [9, 10, 11, 12, 13]
Day category: 0
Blokų: 1921
Input shape: (1921, 45)
Output shape: (1921, 45)
Datos: 2012-01-09 00:00:00 — 2025-12-30 00:00:00
Valanda 9: prognozė=48.7 µg/m³, realiai=49.0 µg/m³
Valanda 10: prognozė=48.8 µg/m³, realiai=47.0 µg/m³
Valanda 11: prognozė=45.2 µg/m³, realiai=33.0 µg/m³
Valanda 12: prognozė=35.7 µg/m³, realiai=20.0 µg/m³
Valanda 13: prognozė=25.8 µg/m³, realiai=20.0 µg/m³
